<p style="text-align: right; margin: 0; line-height: 0;"><img width="100px" height="auto" src="../../.admin/img/eth_logo.png"></p>

<h1 style="text-align: center;">05 Mobile AI</h1>
<hr>


## **9. Making Models Smaller**

<div class="alert alert-light">

  **Name:** (add your name here)

  **Matriculation Number:** (add your number here)

</div>

### 9.1 Introduction

The best-known models run in data centres. A frontier model has hundreds of billions of parameters and needs a rack of accelerators to answer a single question. The Raspberry Pi 5 on your desk has 8 GB of memory, no accelerator, and a power budget of a few watts. Every model on it, from `qwen3.5:0.8b` to `lfm2.5`, fits only because it was made small on purpose.

[05.07_Introduction_to_Local_LLMs.ipynb](05.07_Introduction_to_Local_LLMs.ipynb) introduced these models, and [05.08_Hardware_for_Local_LLMs.ipynb](05.08_Hardware_for_Local_LLMs.ipynb) the hardware they run on. This notebook asks what is lost when a model is made small enough for such a device, and which way of making it small loses the least.

Four techniques do nearly all of the work in practice.

| Technique | What it changes | What it costs |
|:----------|:----------------|:--------------|
| **Quantization** | stores every weight with fewer bits | a small, controllable loss of precision |
| **Pruning** | deletes the weights that contribute least | usually needs retraining, and rarely shrinks the file as much as the name suggests |
| **Distillation** | trains a small model to imitate a large one | a full training run |
| **Selective parameter activation** | keeps every weight but uses only a few per token | saves computation, not memory |

We apply these techniques to a real model, push them until the model breaks, and measure what breaks first. Sections 9.2 to 9.7 run on your laptop in a few minutes. Section 9.8 then checks whether the same effects show up on the Raspberry Pi, with one prediction and one measurement of five to ten minutes. Allow about **75 minutes** in total, plus time for the questions.

By the end, you should be able to estimate a model's memory from its parameter count and precision, explain why grouping makes 4-bit quantization work, compare quantization with pruning on size and quality, and measure the memory wall on the Pi.

#### 9.1.1 Environment Setup

Sections 9.2 to 9.7 use `torch` and `sentence-transformers`, and download a small model (about 90 MB) the first time. Section 9.8 needs the Pi, connected as in [05.04_GettingStarted_with_Raspberry_Pi.ipynb](05.04_GettingStarted_with_Raspberry_Pi.ipynb).

<div class="alert alert-block alert-success">

If a library is missing, use the installation instructions in [00.1_Check_Course_Environment.ipynb](../00_Intro/00.1_Check_Course_Environment.ipynb). Select that Python environment as this notebook's kernel.
</div>

#### 9.1.2 Why size matters

Size matters for three different reasons.

The first is memory. The weights must fit into RAM next to the operating system and everything else running on the device. If they do not fit, the model does not load at all, or the whole system becomes unusable.

The second is bandwidth. To write a single token, the processor has to read every active weight from memory once, so generation is limited by how fast memory can be read, not by how fast the processor can multiply. [05.08](05.08_Hardware_for_Local_LLMs.ipynb) measured this on the Pi:

$$
\text{tokens per second} \;\approx\; \frac{\text{memory bandwidth}}{\text{bytes of active weights}}
$$

Halving the number of bytes roughly doubles the generation speed. On a device like the Pi, compression is the main way to make a model faster, not only smaller.

The third is energy. Moving a byte from memory to the processor costs much more energy than the arithmetic done on it. Fewer bytes mean a longer battery life and, on a passively cooled device, less throttling.

On a mobile device, the size of a model decides its storage, its speed and its battery use at the same time. Choosing a smaller model is never only a decision about quality.

### 9.2 Where the Memory Goes

It starts with a simple multiplication:

$$
\text{weight memory} \;=\; \text{number of parameters} \;\times\; \text{bytes per parameter}
$$

The second factor is the one we can change. Full precision is 32-bit floating point (`FP32`), 4 bytes per weight. Today's large models are trained and served mostly in 16-bit formats, and everything smaller is a compression decision:

| Format | Bytes per weight | Typical role |
|:-------|:----------------:|:-------------|
| `FP32` | 4 | full precision, kept as a master copy of the weights during training |
| `FP16` / `BF16` | 2 | the format large models are trained and served in |
| `INT8` | 1 | comfortable on-device format, barely worse in quality |
| `INT4` | 0.5 | the format most models on your Pi use |

When Ollama labels a model `Q4_K_M`, the `4` is this number: four bits per weight. Section 9.3 explains the `K_M`.

Let us compute the table for the models on your Pi. The Raspberry Pi 5 has 8 GB, of which roughly 2 GB are taken by the operating system and the runtime, so about 6 GB is realistically available.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BYTES_PER_WEIGHT = {"FP32": 4.0, "FP16": 2.0, "INT8": 1.0, "INT4": 0.5}

# Parameter counts of models on your Pi (caspi.models(info=True) lists them), plus two data-center models.
MODELS = {
    "qwen3.5:0.8b":        0.8e9,
    "granite4.2:3b":       3.7e9,
    "qwen3.5:4b":          4.2e9,
    "qwen2.5-coder:7b":    7.6e9,
    "lfm2.5":              8.5e9,
    "qwen3.5:9b":          9.0e9,
    "llama3.1:70b":       70.0e9,
    "a frontier model":  600.0e9,
}

PI_MEMORY_BUDGET_GB = 6.0   # 8 GB board, roughly 2 GB reserved for the operating system


def weight_memory_gb(n_parameters, precision):
    return n_parameters * BYTES_PER_WEIGHT[precision] / 1e9


footprint = pd.DataFrame(
    {precision: [weight_memory_gb(n, precision) for n in MODELS.values()]
     for precision in BYTES_PER_WEIGHT},
    index=list(MODELS),
)
footprint.index.name = "model"

print("Memory needed for the weights, in GB\n")
print(footprint.round(2))

print(f"\n\nDoes it fit into {PI_MEMORY_BUDGET_GB:.0f} GB of usable Raspberry Pi memory?\n")
print(footprint.map(lambda gb: "yes" if gb <= PI_MEMORY_BUDGET_GB else "-"))

# The same table as a picture: one row per model, one dot per precision.

PRECISION_COLORS = {"FP32": "#c7e9c0", "FP16": "#74c476", "INT8": "#238b45", "INT4": "#00441b"}

fig, ax = plt.subplots(figsize=(10, 4.8))
for precision, color in PRECISION_COLORS.items():
    ax.scatter(footprint[precision], range(len(footprint)), s=80, color=color,
               edgecolor="white", linewidth=0.8, label=precision, zorder=3)

ax.axvline(PI_MEMORY_BUDGET_GB, color="#d62728", linewidth=1.2, linestyle="--", zorder=2)
ax.text(PI_MEMORY_BUDGET_GB * 1.15, 0.2, "usable Pi memory (6 GB)", color="#d62728", fontsize=9)

ax.set_xscale("log")
ax.set_yticks(range(len(footprint)), footprint.index)
ax.set_xlabel("memory needed for the weights (GB, log scale)")
ax.invert_yaxis()
ax.grid(axis="x", color="#dddddd", linewidth=0.6, zorder=0)
ax.legend(title="precision", loc="lower right")
ax.set_title("Every dot left of the line fits on the Raspberry Pi", loc="left")
fig.tight_layout()
plt.show()

The chart is the table drawn out: each dot is one model at one precision, and the red line is the usable memory. Read it by row and by column. Along a row, moving from `FP32` to `INT4` divides the memory by eight, and that is what turns `qwen3.5:9b` from an impossible 36 GB into a 4.5 GB model that fits. Along a column, note where the "yes" turns into a "-": at `INT4` the Pi can hold a 9-billion-parameter model, while at `FP32` it cannot even hold `granite4.2:3b`. On the same hardware, the compression decision lets you run a model eight times larger.

The frontier model is there for proportion: no amount of quantization brings it onto this device. A model of that size has to run in the cloud.

#### 9.2.1 The context window is not free either

The weights are not the only thing in memory. While a model generates, it also keeps a **KV cache**: for every token it has already seen, it stores the key and value vectors of every layer so it does not have to recompute them. That cache grows linearly with the length of the conversation:

$$
\text{cache bytes} \;=\; 2 \;\times\; \text{layers} \;\times\; \text{KV heads} \;\times\; \text{head dimension} \;\times\; \text{tokens} \;\times\; \text{bytes per value}
$$

The leading 2 counts the keys and the values. This is the cost behind the `num_ctx` setting (see [05.12](05.12_Understanding_Inference_Control.ipynb)): a larger context window takes memory that the weights could otherwise use.

In [ ]:
def kv_cache_mb(n_layers, n_kv_heads, head_dim, n_tokens, bytes_per_value=2):
    """Memory the model spends on remembering the conversation so far."""
    return 2 * n_layers * n_kv_heads * head_dim * n_tokens * bytes_per_value / 1e6


# The shape of granite4.2:3b, from caspi.ollama("/api/show", {"model": "granite4.2:3b"})["model_info"]:
# 40 layers, 8 key/value heads, and 2560 / 40 heads = 64 dimensions per head.
N_LAYERS, N_KV_HEADS, HEAD_DIM = 40, 8, 64

for n_tokens in (512, 2048, 4096, 8192):
    cache = kv_cache_mb(N_LAYERS, N_KV_HEADS, HEAD_DIM, n_tokens)
    print(f"num_ctx = {n_tokens:5d} tokens -> {cache:7.1f} MB of KV cache")

Our Pis run with a context of 8192 tokens. For `granite4.2:3b` that is about 670 MB of cache, which explains most of the gap between its 2.2 GB file and the 3.1 GB that `caspi.ps()` reports once it is loaded.

### 9.3 Quantization: Fewer Bits per Weight

Quantization is the technique used in practice: every model on your Raspberry Pi is quantized.

#### 9.3.1 What a trained weight matrix looks like

Quantization works because of how trained weights are distributed: not evenly over their range, but in a narrow bell around zero, with a few outliers far away. If the values were spread evenly, we would need every bit of precision. Because they cluster, a coarse grid of a few well-placed values keeps nearly all of the information.

We work with `all-MiniLM-L6-v2`, the sentence-embedding model from [05.02](05.02_Understanding_Word_Embedding.ipynb) and the same model as `all-minilm` on your Pi. It is small enough to compress many times in one notebook, and its job is easy to score automatically: it puts related words near each other.

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

torch.set_num_threads(4)

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")   # also on laptops with a GPU: later cells read the weights with numpy
encoder = model[0].auto_model

# Keep an untouched copy so that every experiment can start from the original model.
ORIGINAL_WEIGHTS = {name: p.detach().clone() for name, p in encoder.named_parameters()}
N_PARAMETERS = sum(p.numel() for p in encoder.parameters())


def is_matrix(name, parameter):
    """The large 2-D weight matrices: what every real compression pipeline touches."""
    return parameter.dim() == 2 and parameter.numel() >= 4096


def is_linear(name, parameter):
    """Matrices inside the transformer layers, excluding the vocabulary lookup tables."""
    return is_matrix(name, parameter) and "embeddings" not in name


def restore_original():
    """Undo every compression experiment."""
    with torch.no_grad():
        for name, parameter in encoder.named_parameters():
            parameter.copy_(ORIGINAL_WEIGHTS[name])


in_matrices = sum(p.numel() for n, p in encoder.named_parameters() if is_matrix(n, p))
in_layers = sum(p.numel() for n, p in encoder.named_parameters() if is_linear(n, p))

print(f"parameters in total        : {N_PARAMETERS/1e6:6.1f} M")
print(f"held in large 2-D matrices : {in_matrices/1e6:6.1f} M  ({in_matrices/N_PARAMETERS:.1%} of the model)")
print(f"   inside transformer layers : {in_layers/1e6:6.1f} M")
print(f"   inside lookup tables      : {(in_matrices-in_layers)/1e6:6.1f} M")
print(f"stored as FP32             : {N_PARAMETERS*4/1e6:6.1f} MB")

Almost the whole model sits in a few two-dimensional matrices, so compression targets those. The biases and normalisation parameters are sensitive to rounding and too small to matter. In a model this size, about half of the parameters are the vocabulary lookup table, not the transformer layers: for small models, the vocabulary is a large part of the cost.

#### 9.3.2 Rounding onto a grid

Quantization with `b` bits means we are allowed $2^b$ distinct values. The simplest and most common scheme is **symmetric absmax quantization**: find the largest magnitude in the matrix, spread the available levels evenly from `-max` to `+max` and round every weight to the nearest one. We store the small integers plus one floating-point `scale` and multiply back when the weight is needed.

With 4 bits we get the integers −7 to +7. The whole method is three lines of arithmetic:

In [ ]:
def quantize_dequantize(W, bits, group_size=None):
    """Round a weight matrix onto a coarse grid, then decode it back to floating point.

    group_size=None  -> one shared scale for the entire matrix
    group_size=128   -> one scale for every 128 neighbouring weights
    """
    levels = 2 ** (bits - 1) - 1                 # 4 bits -> the integers -7 ... +7
    groups = W.reshape(-1, group_size) if group_size else W.reshape(1, -1)
    scale = groups.abs().amax(dim=1, keepdim=True).clamp(min=1e-12) / levels
    codes = torch.clamp(torch.round(groups / scale), -levels, levels)
    return (codes * scale).reshape(W.shape)


EXAMPLE = "encoder.layer.0.intermediate.dense.weight"
W = ORIGINAL_WEIGHTS[EXAMPLE]

print(f"{EXAMPLE}   shape {tuple(W.shape)}")
print(f"  standard deviation : {W.std():.4f}")
print(f"  largest magnitude  : {W.abs().max():.4f}  "
      f"({W.abs().max()/W.std():.0f} standard deviations away from zero)\n")

for bits in (8, 4):
    for group_size in (None, 128, 32):
        recovered = quantize_dequantize(W, bits, group_size)
        error = (recovered - W).abs().mean() / W.abs().mean()
        collapsed = (recovered == 0).float().mean()
        label = "per-tensor" if group_size is None else f"group of {group_size}"
        print(f"  {bits}-bit, {label:14s} -> average error {error:6.1%}, "
              f"{collapsed:5.1%} of weights rounded to zero")

The 8-bit rows look harmless. The 4-bit per-tensor row does not: the error is about as large as the weights themselves, and most of the matrix has been rounded to exactly zero. Something in this matrix breaks the method.

#### 9.3.3 Why a single scale fails and what groups fix

The cause is in the numbers above: the largest weight in the matrix sits more than twenty standard deviations from zero. Absmax quantization has to stretch its grid out to that outlier, so the spacing between grid points becomes wider than the bell that holds almost every weight. All of those weights round to the nearest grid point, which is zero.

The fix is to stop sharing one scale across half a million weights. **Group-wise quantization** gives each block of neighbouring weights (typically 32, 64 or 128 of them) its own scale. An outlier then only damages its own group and every other group gets a grid matched to its own range. This is what the `K` in Ollama's `Q4_K_M` refers to: a block-wise 4-bit format, not a single scale for the whole tensor. The `M`, for medium, means that some tensors get more than four bits.

The price is small: one extra scale per group. At a group size of 128, that adds 16 bits per 128 weights, or 0.125 bits per weight, and it shows up in the file sizes below.

In [ ]:
GROUP_SIZE = 128
BITS = 4
levels = 2 ** (BITS - 1) - 1

all_weights = W.flatten().numpy()
step_tensor = float(W.abs().max()) / levels
grid_tensor = np.arange(-levels, levels + 1) * step_tensor

one_group = W.reshape(-1, GROUP_SIZE)[0].numpy()
step_group = float(np.abs(one_group).max()) / levels
grid_group = np.arange(-levels, levels + 1) * step_group

rounded_to_zero = float(np.mean(np.abs(all_weights) < step_tensor / 2))
outliers = int(np.sum(np.abs(all_weights) > 0.5))

print(f"{all_weights.size:,} weights in this matrix, of which {outliers} are larger than 0.5")
print(f"the largest of them is {np.abs(all_weights).max():.2f} and it alone sets the grid spacing")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.4), constrained_layout=True)

axes[0].hist(all_weights, bins=300, color="#4c78a8")
for level in grid_tensor:
    axes[0].axvline(level, color="#d62728", linestyle="--", linewidth=0.9)
axes[0].set_yscale("log")
axes[0].set_title(
    f"One scale for the whole matrix: step = {step_tensor:.3f}\n"
    f"{rounded_to_zero:.0%} of all weights fall inside the first step and become zero",
    loc="left", fontsize=10)
axes[0].set_xlabel("weight value")
axes[0].set_ylabel("number of weights (log scale)")

axes[1].hist(one_group, bins=40, color="#4c78a8")
for level in grid_group:
    axes[1].axvline(level, color="#1b9e77", linestyle="--", linewidth=0.9)
axes[1].set_title(
    f"One scale per group of {GROUP_SIZE}: step = {step_group:.3f}\n"
    f"the same 15 levels now have to cover only this group",
    loc="left", fontsize=10)
axes[1].set_xlabel("weight value")
axes[1].set_ylabel("number of weights")

plt.show()

The vertical scale of the left panel is logarithmic, otherwise the few weights near ±1.5 would not be visible at all. Only 15 of more than half a million weights are larger than 0.5, yet they decide where all fifteen red grid lines go, and the first grid interval swallows the whole bell.

The right panel shows one group of 128 weights with its own scale. The same fifteen levels now sit where the weights are. This holds well beyond quantization: a scheme scaled to the extreme values serves the typical values badly.

#### 9.3.4 Does the compressed model still do its job?

The error tells us how far the weights moved, not whether the model still works. A network can absorb a lot of noise in its weights, up to a point where it suddenly cannot. So we measure the model on its task.

Our probe is a vocabulary of one hundred words. From the uncompressed model we record, for every word, its five nearest neighbours in embedding space. Then we compress the model and ask two questions:

- **Agreement with the original model:** the cosine similarity between a word's embedding before and after compression. A value of 1.00 means the compressed model produced exactly the same vector.
- **Top-5 neighbours kept:** the fraction of each word's original five neighbours that are still among its five nearest neighbours. It answers the practical question: would the search results change?

The second measure is strict. It ignores how far the vectors moved and only asks whether the ranking a user would see stayed the same.

In [ ]:
VOCAB = [
    "father", "mother", "son", "daughter", "brother", "sister", "uncle", "aunt",
    "king", "queen", "prince", "princess", "duke", "duchess", "emperor", "empress",
    "doctor", "nurse", "engineer", "teacher", "lawyer", "chef", "scientist", "artist",
    "pilot", "farmer", "soldier", "musician",
    "paris", "london", "zurich", "bern", "geneva", "berlin", "tokyo", "rome",
    "france", "japan", "germany", "switzerland", "italy", "spain", "china", "brazil",
    "car", "bicycle", "train", "airplane", "truck", "motorcycle", "helicopter", "boat",
    "router", "antenna", "battery", "processor", "sensor", "satellite", "modem", "transistor",
    "apple", "orange", "banana", "grape", "pizza", "pasta", "sushi", "bratwurst",
    "bread", "cheese", "coffee", "chocolate",
    "cat", "dog", "horse", "rabbit", "eagle", "salmon", "elephant", "wolf",
    "river", "mountain", "forest", "ocean", "desert", "island", "valley", "glacier",
    "good", "bad", "cold", "hot", "fast", "slow", "heavy", "light",
    "happy", "sad", "loud", "quiet",
]

TOP_K = 5


def embed_vocabulary():
    return model.encode(VOCAB, normalize_embeddings=True, show_progress_bar=False)


def nearest_neighbours(embeddings, k=TOP_K):
    similarity = embeddings @ embeddings.T
    np.fill_diagonal(similarity, -np.inf)      # a word is not its own neighbour
    return np.argsort(-similarity, axis=1)[:, :k]


restore_original()
REFERENCE_EMBEDDINGS = embed_vocabulary()
REFERENCE_NEIGHBOURS = nearest_neighbours(REFERENCE_EMBEDDINGS)


def evaluate_current_model():
    """Compare whatever is currently loaded against the uncompressed reference."""
    embeddings = embed_vocabulary()
    agreement = float(np.mean(np.sum(embeddings * REFERENCE_EMBEDDINGS, axis=1)))
    neighbours = nearest_neighbours(embeddings)
    kept = float(np.mean([len(set(new) & set(old)) / TOP_K
                          for new, old in zip(neighbours, REFERENCE_NEIGHBOURS)]))
    return agreement, kept


print(f"{len(VOCAB)} words embedded. Neighbours found by the uncompressed model:\n")
for word in ("king", "zurich", "antenna", "pizza", "glacier", "cold"):
    i = VOCAB.index(word)
    print(f"  {word:10s} -> {', '.join(VOCAB[j] for j in REFERENCE_NEIGHBOURS[i])}")

The lists make sense: `king` finds royalty, `zurich` finds Switzerland and other cities, `antenna` finds satellite, modem and router. A compressed model should still produce lists like these.

Now we compress the whole model and measure it. For a fair file size, we count the scales as well as the weights, and keep the parameters we do not quantize at 16 bits.

In [ ]:
def quantized_size_mb(bits, group_size=None):
    """File size when the large matrices use `bits` and everything else stays at 16 bits."""
    total_bits = 0
    for name, parameter in encoder.named_parameters():
        if is_matrix(name, parameter):
            total_bits += parameter.numel() * bits
            if group_size:
                total_bits += parameter.numel() // group_size * 16   # one FP16 scale per group
        else:
            total_bits += parameter.numel() * 16
    return total_bits / 8 / 1e6


def apply_quantization(bits, group_size=None):
    restore_original()
    with torch.no_grad():
        for name, parameter in encoder.named_parameters():
            if is_matrix(name, parameter):
                parameter.copy_(quantize_dequantize(ORIGINAL_WEIGHTS[name], bits, group_size))


restore_original()
agreement, kept = evaluate_current_model()
results = [dict(technique="none", setting="FP16 (baseline)", MB=quantized_size_mb(16),
                agreement=agreement, neighbours_kept=kept)]

for bits, group_size in [(8, None), (8, 128), (4, None), (4, 128), (4, 32), (3, 128), (2, 128)]:
    apply_quantization(bits, group_size)
    agreement, kept = evaluate_current_model()
    results.append(dict(
        technique="quantization",
        setting=f"INT{bits} " + ("per-tensor" if group_size is None else f"group={group_size}"),
        MB=quantized_size_mb(bits, group_size),
        agreement=agreement,
        neighbours_kept=kept,
    ))

restore_original()
print(pd.DataFrame(results).set_index("setting").round(3))

Eight bits is essentially free: with groups, the model is half the size, agrees with the original almost perfectly, and keeps nearly every neighbour ranking.

At four bits there is a real trade-off. `INT4 group=128` is about a quarter of the FP16 baseline and still keeps most rankings. It is no longer free, but the loss is acceptable for most uses, which is why your Pi runs this format.

How the bits are spent matters as much as how many there are. `INT4 per-tensor` and `INT4 group=128` differ in size by about three percent, yet one works and the other does not. Smaller groups cost bytes too: `group=32` stores four times as many scales and is about 9 % larger than `group=128`, for a little more quality. Whether that is worth it, only a measurement on your own model can tell.

Below 4 bits, quality does not decline smoothly: it holds up for a while and then collapses. That is why compression levels have to be measured, as you cannot interpolate between them.

#### 9.3.5 Quantizing after training, or training for quantization

Everything above is **post-training quantization** (PTQ): take a finished model, round its weights, ship it. It needs no data and runs in minutes, which is why it dominates in practice. **Quantization-aware training** (QAT) simulates the rounding during training, so the optimizer sees the damage and learns weights that survive the grid. It costs a training run, but it recovers much of the quality lost at 4 bits and below.

Your Pi carries a QAT model: `-it-qat` in `gemma4:e2b-it-qat` means *instruction-tuned, quantization-aware trained*. Google trained it knowing it would be quantized, so it does better than a plain 4-bit conversion of the same network. The same model at standard precision, `gemma4:e2b`, is a 7.2 GB download: more than the 6 GB the Pi has to offer, as the table in Section 9.2 predicts.

### 9.4 Pruning: Fewer Weights

Quantization keeps every weight and describes each one more cheaply. **Pruning** deletes the weights that seem to contribute least, usually the ones closest to zero. It comes in two kinds:

- **Unstructured pruning** zeroes individual weights wherever they sit, so it can always pick exactly the least useful ones.
- **Structured pruning** removes whole neurons, entire rows of a matrix, and with them useful weights that happen to share a row with useless ones.

The cell measures both in the same way as before, on the transformer layers only: deleting rows of the vocabulary table would simply delete words.

In [ ]:
def prune(W, sparsity, structured):
    """Zero the smallest weights; with structured=True, the weakest whole rows instead."""
    pruned = W.clone()
    if structured:
        weakest = torch.argsort(W.norm(dim=1))[:round(sparsity * W.shape[0])]
        pruned[weakest] = 0.0
    else:
        threshold = W.abs().flatten().kthvalue(round(sparsity * W.numel())).values
        pruned[W.abs() <= threshold] = 0.0
    return pruned


def pruned_size_mb(sparsity, structured):
    """The weights that are left, at 16 bits each. Unstructured pruning also needs
    one bit per weight to record where the holes are."""
    bits = 0
    for name, parameter in encoder.named_parameters():
        if is_linear(name, parameter):
            bits += parameter.numel() * ((1 - sparsity) * 16 + (0 if structured else 1))
        else:
            bits += parameter.numel() * 16
    return bits / 8 / 1e6


for sparsity, structured in [(0.3, False), (0.5, False), (0.7, False), (0.9, False),
                             (0.1, True), (0.25, True), (0.5, True)]:
    restore_original()
    with torch.no_grad():
        for name, parameter in encoder.named_parameters():
            if is_linear(name, parameter):
                parameter.copy_(prune(ORIGINAL_WEIGHTS[name], sparsity, structured))
    agreement, kept = evaluate_current_model()
    results.append(dict(technique="pruning (structured)" if structured else "pruning (unstructured)",
                        setting=f"{sparsity:.0%} of " + ("neurons" if structured else "weights"),
                        MB=pruned_size_mb(sparsity, structured), agreement=agreement, neighbours_kept=kept))

restore_original()
pruning_results = pd.DataFrame(results).set_index("setting")
print(pruning_results[pruning_results["technique"].str.startswith("pruning")].round(3))

Unstructured pruning degrades gracefully, structured pruning does not. Removing 30 % of individual weights keeps more rankings intact than removing 10 % of whole neurons, and gives a smaller file: a neuron that looks weak on average can still carry a few weights the network depends on.

Removing half of the weights does not halve the file: recording where the holes are costs about one bit per weight. And unless the software skips the zeros, the multiplication reads them like any other number, so on a Pi a model that is 50 % sparse is as slow as the original. Real pipelines fine-tune the model after pruning and win back much of the lost quality, a step our quick measurement skipped.

### 9.5 Quality against Size: Putting the Decisions on One Chart

All measurements are now in one table. The chart places each setting at its size and its quality, so good settings sit at the upper left. A point that has another point above and to its left is never worth choosing: the other setting is both smaller and better.

In [ ]:
comparison = pd.DataFrame(results)

COLORS = {
    "none": "#757575",
    "quantization": "#1b9e77",
    "pruning (unstructured)": "#d95f02",
    "pruning (structured)": "#7570b3",
}

# The two 4-bit group settings land almost on top of each other; nudge one label apart.
LABEL_OFFSETS = {"INT4 group=32": (8, -14)}

fig, ax = plt.subplots(figsize=(11, 6))

for technique, color in COLORS.items():
    group = comparison[comparison["technique"] == technique].sort_values("MB")
    if technique != "none":
        # Connect the settings that form one continuous trade-off curve.
        curve = group[~group["setting"].str.contains("per-tensor")]
        ax.plot(curve["MB"], curve["neighbours_kept"], color=color, linewidth=1.2, alpha=0.55)
    ax.scatter(group["MB"], group["neighbours_kept"], s=95, color=color,
               label=technique, edgecolor="white", zorder=3)
    for _, row in group.iterrows():
        ax.annotate(row["setting"], (row["MB"], row["neighbours_kept"]),
                    textcoords="offset points",
                    xytext=LABEL_OFFSETS.get(row["setting"], (8, 6)), fontsize=8)

ax.annotate("better", xy=(5.0, 1.03), fontsize=10, color="#555555")
ax.annotate("", xy=(2.0, 1.07), xytext=(9.5, 0.98),
            arrowprops=dict(arrowstyle="->", color="#555555"))

ax.axhline(1.0, color="#757575", linewidth=0.8, linestyle=":")
ax.set_xlabel("stored size of the model (MB)")
ax.set_ylabel(f"top-{TOP_K} neighbours kept")
ax.set_xlim(0, comparison["MB"].max() * 1.15)
ax.set_ylim(0, 1.12)
ax.set_title("Quality against size: every point is one compression decision", loc="left")
ax.legend(loc="lower right")

plt.show()

`INT4 group=128` is about three times smaller than 50 % unstructured pruning and keeps more of the rankings, and `INT8 group=128` beats every pruning setting while staying essentially lossless. Only the broken settings from Section 9.3, `INT4 per-tensor` and `INT2`, fall below the pruning curve.

This is one small encoder and one task, and our pruning was not followed by fine-tuning, so the exact positions of the dots are not a general law. The overall picture holds more widely: **on memory-bound devices, fewer bits per weight beat fewer weights.** That is why models for phones and small devices are quantized, and why Ollama reports a quantization level rather than a sparsity. In industry the techniques are combined, often pruning and distillation first, and quantization almost always comes last [02, 12].

### 9.6 Distillation: Training a Small Model to Imitate a Large One

Quantization and pruning both start from a finished model and remove something from it. **Knowledge distillation** takes a different route: it trains a new, smaller model (the *student*) from scratch to reproduce the behaviour of a large one (the *teacher*).

The difference lies in what the student is trained on. Ordinary training uses **hard labels**: the correct next word and nothing else. Distillation uses the teacher's whole **output distribution** as the target, its probability for every possible next token. That distribution holds much more information than the single correct answer, sometimes called *dark knowledge*: what is right, what was nearly right, and what was never plausible.

To make that information usable, the teacher's distribution is softened with a temperature, the same temperature as in [05.05](05.05_Introduction_to_LLMs.ipynb). At `T = 1` a confident teacher puts almost all of its probability on one token, and the runners-up are close to zero. A higher temperature lifts them high enough to influence the student's training.

In [ ]:
CANDIDATES = ["battery", "signal", "price", "resolution", "temperature", "elephant"]
teacher_logits = np.array([6.0, 4.6, 3.4, 3.0, 1.4, -3.0])   # scores from a large teacher model


def softmax(scores, temperature=1.0):
    shifted = scores / temperature
    shifted = shifted - shifted.max()
    weights = np.exp(shifted)
    return weights / weights.sum()


hard_label = np.zeros(len(CANDIDATES))
hard_label[CANDIDATES.index("battery")] = 1.0
teacher_t1 = softmax(teacher_logits, temperature=1.0)
teacher_t4 = softmax(teacher_logits, temperature=4.0)

targets = pd.DataFrame(
    {"hard label": hard_label, "teacher, T=1": teacher_t1, "teacher, T=4": teacher_t4},
    index=CANDIDATES,
)
print('Prompt: "The mobile phone has a low ..."\n')
print(targets.round(4))

fig, ax = plt.subplots(figsize=(10, 4))
offsets = np.arange(len(CANDIDATES))
for shift, column, color in [(-0.27, "hard label", "#757575"),
                             (0.00, "teacher, T=1", "#4c78a8"),
                             (0.27, "teacher, T=4", "#1b9e77")]:
    ax.bar(offsets + shift, targets[column], width=0.27, label=column, color=color)

ax.set_xticks(offsets, CANDIDATES)
ax.set_ylabel("probability")
ax.set_title('What the student is asked to reproduce: "The mobile phone has a low ..."', loc="left")
ax.legend()

plt.show()

The hard label says: the answer is `battery`, and everything else is equally wrong. The teacher knows more: `signal` and `price` are reasonable continuations, `resolution` is plausible, `elephant` is not. At `T = 1` that ranking is there, but its tail is tiny. At `T = 4` every candidate reaches the student. The student learns how the words relate to each other, not only which one is right, and needs far fewer examples than the teacher did.

Many capable small models are made this way. For example, `deepseek-r1:1.5b`, is not a shrunken DeepSeek-R1 but a small Qwen model trained on reasoning written by the large one. Distillation needs a teacher, a corpus and a full training run, so it is done by whoever builds a model. For you, it is a choice of model, not a setting.

### 9.7 Selective Parameter Activation: Using Fewer Weights at a Time

The last technique does not make a model smaller but it makes each token cheaper. In a **mixture-of-experts** architecture, each layer holds many parallel sub-networks (*experts*), and a small router picks a few of them for each token. Since generation speed is set by the bytes read per token, using only a fraction of the weights makes generation much faster.

The catch is memory: the router may pick any expert for the next token, so **every expert has to be in memory**. A mixture of experts needs the memory of all its parameters and runs at the speed of the active ones. In a data centre, where memory is plentiful, that is a good trade. On a Pi it helps only if the whole model fits, as `lfm2.5` does: 8.5 billion parameters in a 5.2 GB file, of which 4 of 32 experts per layer, about 1.5 billion parameters, are used for each token. Section 9.8 measures what that does to its speed.

`gemma4:e2b-it-qat` takes another route to the same goal. The `e2b` stands for *effective 2B*: the model holds more parameters than that, but is designed to run with the memory and speed of a model of about 2 billion.

### 9.8 Checking the Story on the Raspberry Pi

Everything so far was measured on a small encoder on your laptop. Two claims can be checked on the Pi itself:

1. Generation is limited by memory: every token reads all the weights once, so a larger file means slower generation.
2. A mixture of experts breaks that rule, because it reads only the experts it uses.

In [ ]:
import caspi

caspi.connect()

#### 9.8.1 Check the rule on three models

05.08 found that writing an answer is limited by memory: a dense model reads its whole file for every token. The next cells check that rule on three models, one model per cell: `qwen3.5:0.8b` and `qwen3.5:4b`, two sizes of one dense family, and `lfm2.5`, the mixture of experts from Section 9.7. `measure` frees the memory with `caspi.stop()`, loads one model, records its file size and generation speed, and frees the memory again. `lfm2.5` alone takes more than 5 GB of the Pi's 8: do the same whenever you load a large model yourself (05.04, Section 4.5.2).

**Make a prediction before you run it** as Question 07 asks for it. `qwen3.5:4b` is a file about 2.5 times larger than `qwen3.5:0.8b`, and `lfm2.5` about 4 times. If every token has to read all of a model's weights, how fast should each of them be, relative to the smallest? Together the cells take five to ten minutes, most of it for loading the models off the SD card.

<div class="alert alert-block alert-info">

While it runs, watch the memory. Open a terminal on your laptop, log in with `ssh cas@172.31.250.1` as in Section 4.7 of [05.04](05.04_GettingStarted_with_Raspberry_Pi.ipynb), and start `htop` on the Pi.

Watch the `Mem` bar as each model loads. `lfm2.5` uses only a few of its experts for each token, but all of them have to be in memory.
</div>

In [ ]:
QUESTION = ("A train leaves Zurich at 14:10 and arrives in Geneva at 16:56. "
            "How long is the trip? Explain your calculation in two sentences.")
file_gb = {entry["name"]: entry["size_gb"] for entry in caspi.models(info=True)}
speeds, answers = {}, {}


def measure(model):
    """Load one model alone, ask the question, note its speed, and free the memory again."""
    caspi.stop()   # load each model alone: two large ones can freeze the Pi
    caspi.load(model)
    reply = caspi.ask(model, QUESTION, think=False, temperature=0, seed=1, show=False)
    caspi.stop(model)
    answers[model] = reply.text
    speeds[model] = {"file_gb": file_gb[model], "tokens_per_s": round(reply.stats["reply_tokens_per_s"], 1)}
    print(f"{model}: {speeds[model]['tokens_per_s']} tokens per second")

In [ ]:
measure("qwen3.5:0.8b")

In [ ]:
measure("qwen3.5:4b")

In [ ]:
measure("lfm2.5:latest")

In [ ]:
pi = pd.DataFrame(speeds).T
pi["GB read per second"] = (pi["file_gb"] * pi["tokens_per_s"]).round(1)
pi

Compare the table with your prediction. If every token reads the whole file once, file size times tokens per second is the amount of memory read per second. For the two dense models that number should come out about the same: it is what the Pi's memory can deliver, whichever model reads from it. Only roughly, though: each `qwen3.5` file also holds a vision encoder of about half a gigabyte (05.07), which is not read while the model writes text, so both numbers come out too high, the 0.8b's the most. This is the memory wall, and it is why a 4-bit model is faster than a 16-bit one, not only smaller.

`lfm2.5` lands far above that line, at a rate the Pi's memory cannot deliver at all. It does not have to: for each token it reads only 4 of the 32 experts in each layer. It needs the memory of all its parameters, but it generates at the speed of the active ones.

#### 9.8.2 What the size bought

The same three answers also show what each model can do. The trip takes 2 hours and 46 minutes. Read the explanations as well, not only the final number:

In [ ]:
for model, answer in answers.items():
    print(f"========== {model}\n{answer}\n")

A model can reach the right number with a wrong explanation, and a small model can be right where a larger one is not. A larger model pays in memory and in time for every token it writes. Whether that price is worth it depends on the task: for safety-relevant answers you would choose a larger model than for autocomplete suggestions.

#### 9.8.3 Matching the technique to the model

Every technique in this notebook can be found in real models. Which one made each of these models small?

| Model | Technique that made it small | How you can tell |
|:------|:-----------------------------|:-----------------|
| `gemma4:e2b-it-qat` | quantization, trained for it (QAT) | the `qat` suffix, Ollama reports a 4-bit format |
| `lfm2.5` | selective parameter activation (mixture of experts) | the model description lists 32 experts and 4 used per token, Section 9.8.1 |
| `qwen3.5:0.8b` | quantization, Qwen has not published whether its small models were also distilled, as the small Qwen3 models were [13] | its quantization, the rest is not published |
| `deepseek-r1:1.5b` (not on our Pi) | distillation from a much larger reasoning model | it reasons far better than its size suggests |

<div class="alert alert-block alert-info">

Find the evidence for the rows on your Pi with `caspi.models(info=True)`, `caspi.show()` and `caspi.ollama("/api/show", {"model": ...})["model_info"]`. One row is harder to confirm than the others.
</div>

Pruning is missing from the table. Pruned models are rarely released on their own as it is used inside industrial pipelines as a step before distillation and quantization [12]. Section 9.5 came to the same conclusion from measurements.

### 9.9 Summarizing Remarks

Making a model smaller means choosing between techniques with different costs and different ways of failing. Quantization is the usual choice: it needs no training data, and it makes the file smaller and generation faster at the same time. Eight bits are close to free, four bits are the usual compromise on small devices, and below four bits quality collapses. How the bits are spent matters as much as how many there are: two 4-bit versions of almost the same size differed only in how the scales were grouped, and one of them did not work.

Pruning saves the wrong thing for mobile devices, which are limited by bytes and bandwidth, not by the number of parameters. Distillation and selective activation are decided by whoever builds a model. You use them by choosing which model to download. `lfm2.5` showed both sides of a mixture of experts: fast on the Pi, but only because its full size still fits.

Each technique trades one resource for another: precision for memory, parameters for quality, training compute for inference cost. None of them adds capability. So the useful question is which resource is scarce on your device, and measurements on that device answer it better than settings copied from elsewhere.

### 9.10 Further Reading and Literature

[01] Hinton, G., Vinyals, O., & Dean, J. (2015). **Distilling the Knowledge in a Neural Network.** NeurIPS Deep Learning Workshop. [Paper](https://arxiv.org/abs/1503.02531)

[02] Han, S., Mao, H., & Dally, W. J. (2016). **Deep Compression: Compressing Deep Neural Networks with Pruning, Trained Quantization and Huffman Coding.** ICLR. [Paper](https://arxiv.org/abs/1510.00149)

[03] Shazeer, N., Mirhoseini, A., Maziarz, K., Davis, A., Le, Q., Hinton, G., & Dean, J. (2017). **Outrageously Large Neural Networks: The Sparsely-Gated Mixture-of-Experts Layer.** ICLR. [Paper](https://arxiv.org/abs/1701.06538)

[04] Jacob, B., Kligys, S., Chen, B., Zhu, M., Tang, M., Howard, A., Adam, H., & Kalenichenko, D. (2018). **Quantization and Training of Neural Networks for Efficient Integer-Arithmetic-Only Inference.** CVPR. [Paper](https://arxiv.org/abs/1712.05877)

[05] Frankle, J., & Carbin, M. (2019). **The Lottery Ticket Hypothesis: Finding Sparse, Trainable Neural Networks.** ICLR. [Paper](https://arxiv.org/abs/1803.03635)

[06] Sanh, V., Debut, L., Chaumond, J., & Wolf, T. (2019). **DistilBERT, a distilled version of BERT: smaller, faster, cheaper and lighter.** NeurIPS EMC^2 Workshop. [Paper](https://arxiv.org/abs/1910.01108)

[07] Wang, W., Wei, F., Dong, L., Bao, H., Yang, N., & Zhou, M. (2020). **MiniLM: Deep Self-Attention Distillation for Task-Agnostic Compression of Pre-Trained Transformers.** NeurIPS. [Paper](https://arxiv.org/abs/2002.10957). The model used throughout this notebook is itself a distilled one.

[08] Dettmers, T., Lewis, M., Belkada, Y., & Zettlemoyer, L. (2022). **LLM.int8(): 8-bit Matrix Multiplication for Transformers at Scale.** NeurIPS. [Paper](https://arxiv.org/abs/2208.07339)

[09] Frantar, E., Ashkboos, S., Hoefler, T., & Alistarh, D. (2023). **GPTQ: Accurate Post-Training Quantization for Generative Pre-trained Transformers.** ICLR. [Paper](https://arxiv.org/abs/2210.17323)

[10] Frantar, E., & Alistarh, D. (2023). **SparseGPT: Massive Language Models Can Be Accurately Pruned in One-Shot.** ICML. [Paper](https://arxiv.org/abs/2301.00774)

[11] Lin, J., Tang, J., Tang, H., Yang, S., Dang, X., & Han, S. (2024). **AWQ: Activation-aware Weight Quantization for LLM Compression and Acceleration.** MLSys. [Paper](https://arxiv.org/abs/2306.00978)

[12] Muralidharan, S., Sreenivas, S. T., Joshi, R., Chochowski, M., Patwary, M., Shoeybi, M., Catanzaro, B., Kautz, J., & Molchanov, P. (2024). **Compact Language Models via Pruning and Knowledge Distillation.** NeurIPS. [Paper](https://arxiv.org/abs/2407.14679)

[13] Yang, A., Li, A., Yang, B., et al. (2025). **Qwen3 Technical Report.** [Paper](https://arxiv.org/abs/2505.09388). Its small models were distilled from the larger ones.

---
### 9.11 Questions

<div class="alert alert-block alert-info">Fill in your responses and save this notebook. See the <a href="../../03_Assignments/Submission-Guidelines.md">submission guidelines</a> for how to submit your work.</div>

Use your own outputs as evidence. Copy the relevant numbers into your answer so that they remain visible after saving.

**Question 01: Reading the memory table**

Which is the smallest model in the Section 9.2 table that does *not* fit into the Pi's usable memory at `FP16`, and which precision is the first one that makes it fit?

(answer here)

**Question 02: The cost of context**

Use Section 9.2.1: how much memory does the KV cache of `granite4.2:3b` take at 8192 tokens? Compare it with the gap between the file size and the loaded size of the same model (Question 04 of 05.04).

(answer here)

**Question 03: One scale or many**

Report the file size and the top-5 neighbour score for `INT4 per-tensor`, `INT4 group=128` and `INT4 group=32`. Why does a difference of a few percent in size produce such a different outcome, and was `group=32` worth its extra bytes?

(answer here)

**Question 04: Where quality collapses**

In your run, at how many bits does quality first fall below roughly 60 % of neighbours kept? Is the decline gradual or sudden, and what does that imply about choosing a compression level without measuring?

(answer here)

**Question 05: Pruning or quantization**

Compare 50 % unstructured pruning with `INT4 group=128` using your own numbers for size and quality. Which would you deploy? Why did removing half of the weights not remove half of the megabytes?

(answer here)

**Question 06: Soft labels**

What information does a teacher's softened output distribution contain that a hard label does not? Use one concrete pair of tokens from the Section 9.6 output to make the point.

(answer here)

**Question 07: The memory wall**

What did you predict in Section 9.8.1 for `qwen3.5:4b` and `lfm2.5` relative to `qwen3.5:0.8b`, and what did you measure, in tokens and GB read per second? Explain how `lfm2.5` can be bigger and faster at the same time, and why that only helps when the model fits into memory. If you watched htop: how full did the memory get?

(answer here)

**Question 08: What the size bought**

Which models answered the train question correctly, and were their explanations correct too? Was the extra size worth its cost in speed for this task, and would you choose differently for a safety-critical application?

(answer here)

**Question 09: Evidence for each technique**

Give the evidence you found for each row of the table in Section 9.8.3 that is on your Pi. Which row was hardest to confirm, and why?

(answer here)

**Question 10: Optional: advising a vendor**

A company wants an offline assistant on a handheld device with 4 GB of RAM. Using the evidence from this notebook, state which compression technique you would require of a vendor, which claim you would ask them to demonstrate, and what measurement you would run to verify it.

(answer here)

**OPTIONAL: Did you read through and understand all the tasks in this Notebook? Are there any shortcomings, missing parts, or sections that should be improved?**

 (please help the team by telling us what part was not understood, and we will work on it)